In [10]:
from dataclasses import dataclass
from typing import Any, Dict, List

import mcp.types as types
from mcp.server.fastmcp import FastMCP
from pydantic import BaseModel, ConfigDict, Field, ValidationError

In [4]:
def _search_amazon(query: str, hits: int = 10) -> List[Dict[str, Any]]:
    """
    Scrape Amazon Japan search results for a given keyword.  This function
    accesses the public search page and attempts to parse product information.
    Amazon employs aggressive anti‑scraping measures; therefore requests may
    return HTTP 403 or incomplete content.  If scraping fails, the function
    returns an empty list.  Use a realistic User‑Agent and accept‑language
    header to mimic a browser and improve the chances of success.
    """
    from urllib.parse import quote
    encoded = quote(query)
    url = "https://www.amazon.co.jp/s"
    params = {"k": encoded}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36",
        "Accept-Language": "ja-JP,ja;q=0.9,en-US;q=0.8,en;q=0.7",
    }
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=10)
        if resp.status_code != 200:
            return []
        soup = BeautifulSoup(resp.text, "html.parser")
        # Amazon identifies search result items with data-component-type
        items = soup.select("div.s-result-item[data-component-type='s-search-result']")
        results: List[Dict[str, Any]] = []
        for item in items:
            # Name and link
            h2 = item.find("h2")
            if not h2:
                continue
            a_tag = h2.find("a")
            if not a_tag:
                continue
            name = a_tag.get_text(strip=True)
            href = a_tag.get("href")
            url_ = f"https://www.amazon.co.jp{href}" if href else None
            # Image
            img_el = item.select_one("img.s-image")
            image_url = img_el.get("src") if img_el else None
            # Price: Amazon uses multiple classes; attempt to parse
            price_whole = item.select_one("span.a-price-whole")
            price_fraction = item.select_one("span.a-price-fraction")
            price = None
            if price_whole:
                price_str = price_whole.get_text(strip=True)
                if price_fraction:
                    price_str += price_fraction.get_text(strip=True)
                price_str = price_str.replace(",", "").strip()
                try:
                    price = float(price_str)
                except Exception:
                    price = None
            # Description: Amazon search results do not include a description; leave None
            results.append({
                "name": name,
                "description": None,
                "price": price,
                "rating": None,
                "image": image_url,
                "url": url_,
            })
            if len(results) >= hits:
                break
        return results
    except Exception:
        return []

In [8]:
from requests_html import HTMLSession
from urllib.parse import quote

def _search_amazon(query: str, hits: int = 10):
    encoded = quote(query)
    url = f"https://www.amazon.co.jp/s?k={encoded}"
    session = HTMLSession()
    r = session.get(url)
    r.html.render(timeout=30)  # 让 JavaScript 执行
    items = r.html.find("div.s-result-item[data-component-type='s-search-result']")
    results = []
    for item in items[:hits]:
        name = item.find("h2 a span", first=True)
        link = item.find("h2 a", first=True)
        img = item.find("img.s-image", first=True)
        price = item.find("span.a-price-whole", first=True)
        results.append({
            "name": name.text if name else None,
            "url": f"https://www.amazon.co.jp{link.attrs['href']}" if link else None,
            "image": img.attrs.get("src") if img else None,
            "price": price.text if price else None,
        })
    return results

In [9]:
query = "iphone"
res = _search_amazon(query, hits=10)
print(res)

RuntimeError: Cannot use HTMLSession within an existing event loop. Use AsyncHTMLSession instead.